# Silver — histórico de cadastro, SCD tipo 2 (etapas 3.1 e 3.2)

Segundo arquivo-fonte do pipeline `olist.silver`. Ele **não substitui** nada da Fase 2: lê as views `sellers_validos` (3.1) e `customers_validos` (3.2), que já existem no `01_silver_pipeline`, e grava uma tabela com **uma linha por versão** do cadastro:

- `olist.silver.sellers_historico` (etapa 3.1) · `olist.silver.customers_historico` (etapa 3.2).

A função `registrar_historico` é escrita uma vez e aplicada às duas entidades: a etapa 3.2 só acrescentou `customers` à lista `HISTORICO`.

| Coluna | O que é |
|---|---|
| `seller_id` / `customer_id`, cidade, UF, CEP (e `customer_unique_id`) | o cadastro daquela versão |
| `__START_AT` | `change_ts` em que a versão passou a valer |
| `__END_AT` | `change_ts` em que a próxima versão (ou a exclusão) encerrou esta; **nulo = versão vigente** |

Decisões (`docs/design/fase-3-cdc-scd2.md`): D1 mantém a silver SCD1 e acrescenta o histórico · D2 o `DELETE` **fecha** a versão, sem linha nova · D3 a ordenação é `change_ts` · D4 só os atributos de negócio abrem versão · D5 mesmo pipeline, segundo arquivo-fonte.

**Este arquivo é fonte de pipeline, não notebook comum.** Adicione-o em *Settings → Source code* do pipeline `olist.silver` (o pipeline só enxerga o que está na lista) e use *Dry run* e *Run pipeline* (roteiro na última célula).

```
bronze.<e> ─► <e>_tipado ─► <e>_validos (Fase 2) ─┬─► silver.<e>            (SCD1: estado atual)
                                                  └─► silver.<e>_historico  (SCD2: uma linha por versão)
```


## Configuração

Importa as bibliotecas e lê o catálogo da configuração do pipeline, sem caminho fixo no código.

In [ ]:
# O QUE FAZ: importa o módulo do pipeline e as funções de coluna do Spark e lê o catálogo da
#   configuração do pipeline (a mesma chave do 01_silver_pipeline). Nenhum dado é lido aqui.
#
# IMPORTAÇÕES:
# - from pyspark import pipelines as dp: módulo das Spark Declarative Pipelines (create_streaming_table,
#   create_auto_cdc_flow). Só existe dentro de um pipeline.
# - pyspark.sql.functions as F: funções nativas de coluna; aqui só F.col.
#
# FUNÇÕES NATIVAS:
# - spark.conf.get(chave, padrão): lê um valor da configuração do pipeline (Settings → Configuration).
#   Sem a chave definida, devolve o padrão ("olist").
#
# Cada arquivo-fonte tem seu próprio código Python, então CATALOG é declarado de novo aqui; as
# VIEWS (customers_validos, sellers_validos), essas sim, são compartilhadas entre os arquivos.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = spark.conf.get("olist.catalogo", "olist")

## Quais colunas abrem uma nova versão

`track_history_except_column_list` diz ao AUTO CDC **quais colunas NÃO contam** como mudança. Todas as outras abrem versão quando o valor muda.

| Fora do rastreamento | Por quê |
|---|---|
| `_ingested_at`, `_source_file`, `_source_file_modified_at`, `_run_id` | Uma duplicata reenviada chega com auditoria diferente; se contasse, cada reenvio abriria uma versão falsa. |
| `operation`, `change_sequence`, `change_ts`, `sim_date` | Metadados do feed, não do cadastro. `change_ts` já ordena as versões (`sequence_by`). |
| `*_bruto`, `_corrupt_record`, `_rescued_data` | Colunas técnicas da tipagem; nem vão para a tabela (`except_column_list`). |

O que sobra é o cadastro: `seller_city`, `seller_state`, `seller_zip_code_prefix` (e `customer_unique_id`, `customer_city`... em `customers`).


In [ ]:
# O QUE FAZ: declara quais entidades têm histórico e as listas de colunas técnicas. Nenhum dado
#   é lido aqui.
#
# RECURSOS:
# - tuple de tuplas ((nome, chave),): configuração fixa que não muda; a etapa 3.2 acrescentou
#   ("customers", "customer_id"). Uma tupla de um item leva vírgula no final.
# - list [...]: lista de nomes de colunas, no formato que o AUTO CDC espera.
# - "*" no desempacotamento (usado adiante): [*a, *b] junta duas listas em uma.
HISTORICO = (
    ("sellers", "seller_id"),
    ("customers", "customer_id"),
)

# colunas que só existem por causa da tipagem/quarentena da Fase 2: não vão para o histórico
COLUNAS_TECNICAS = [
    "change_sequence_bruto", "change_ts_bruto", "sim_date_bruto",
    "_corrupt_record", "_rescued_data",
]
# colunas que existem na tabela mas não abrem versão (D4)
AUDITORIA = ["_ingested_at", "_source_file", "_source_file_modified_at", "_run_id"]
METADADOS_DO_FEED = ["operation", "change_sequence", "change_ts", "sim_date"]

## Registro do histórico

`registrar_historico` declara, para cada entidade, a tabela SCD2 e o fluxo que a alimenta. Os dois detalhes que definem o histórico:

- **`sequence_by = change_ts`** (D3): `__START_AT` e `__END_AT` herdam o tipo desta coluna. Com `change_sequence` o intervalo seria "de 1 até 2"; com `change_ts` é um instante, que serve para "como era na data X".
- **`apply_as_deletes = "operation = 'DELETE'"`** (D2): a linha `DELETE` não vira versão; ela **fecha** a versão vigente (`__END_AT` = `change_ts` da exclusão). O histórico prova que o cadastro existiu e até quando.

A silver SCD1 da Fase 2 continua igual; as duas leem a mesma view validada.


In [ ]:
# O QUE FAZ: para cada entidade de HISTORICO, declara a streaming table de destino e o fluxo
#   AUTO CDC tipo 2 que a alimenta a partir da view <e>_validos (já existente no pipeline).
#
# FUNÇÕES DECLARADAS:
# - registrar_historico(nome, chave): cria silver.<nome>_historico (uma linha por versão do cadastro).
#
# FUNÇÕES NATIVAS:
# - dp.create_streaming_table(name, comment): declara a tabela de destino do AUTO CDC.
# - dp.create_auto_cdc_flow(...): aplica as linhas da origem no destino por chave.
#     * target / source: tabela de destino e a view de origem (nome simples = dataset do pipeline).
#     * keys: colunas que identificam o cadastro (seller_id).
#     * sequence_by: coluna que ordena as versões; aqui change_ts (D3).
#     * apply_as_deletes: condição SQL das linhas que são exclusões; no SCD2 ela FECHA a versão
#       vigente em vez de apagar a linha (D2).
#     * stored_as_scd_type=2: cada mudança nas colunas rastreadas fecha a versão antiga e abre uma
#       nova; o destino ganha __START_AT e __END_AT.
#     * except_column_list: colunas da origem que NÃO vão para o destino.
#     * track_history_except_column_list: colunas que existem no destino mas NÃO abrem versão (D4).
# - source recebe o NOME da view (e não spark.readStream.table): o AUTO CDC a lê como stream.
def registrar_historico(nome: str, chave: str) -> None:
    destino = f"{CATALOG}.silver.{nome}_historico"
    dp.create_streaming_table(
        name=destino,
        comment=(
            f"Histórico (SCD2) de {nome}: uma linha por versão do cadastro, com __START_AT e "
            f"__END_AT (nulo = versão vigente). A exclusão fecha a versão. Fonte: {nome}_validos."
        ),
    )
    dp.create_auto_cdc_flow(
        target=destino,
        source=f"{nome}_validos",
        keys=[chave],
        sequence_by=F.col("change_ts"),
        apply_as_deletes=F.expr("operation = 'DELETE'"),
        stored_as_scd_type=2,
        except_column_list=COLUNAS_TECNICAS,
        track_history_except_column_list=[*COLUNAS_TECNICAS, *AUDITORIA, *METADADOS_DO_FEED],
    )


for nome, chave in HISTORICO:
    registrar_historico(nome, chave)

## Como executar e conferir

Este bloco é só roteiro, não é código do pipeline.

**Etapa 3.1 (`sellers`)** — já executada e conferida (3.361 versões · 3.095 chaves · 3.074 abertas · 0 excluídos com versão aberta). Passos:
1. *Settings → Source code* do pipeline `olist.silver`: **adicione** `src/silver/02_silver_historico` à lista (o `01_silver_pipeline` continua nela).
2. *Dry run*. Confira no grafo: `sellers_historico` ligada a `sellers_validos`. Erro de `except_column_list` ou `apply_as_deletes` aparece aqui.
3. *Run pipeline* **normal** (não *Full refresh*): só a tabela nova é calculada; as outras 10 entidades não têm dado novo.

Confira no SQL editor (esperado entre comentários):

```sql
SELECT count(*) AS versoes, count(DISTINCT seller_id) AS chaves,
       count_if(__END_AT IS NULL) AS abertas
FROM olist.silver.sellers_historico;                       -- 3.361 · 3.095 · 3.074

-- vendedores excluídos: nenhuma versão aberta, __END_AT = data da exclusão
SELECT count(*) FROM olist.silver.sellers s
JOIN olist.silver.sellers_historico h USING (seller_id)
WHERE s.operation = 'DELETE' AND h.__END_AT IS NULL;       -- 0

-- "como era o vendedor X em determinada data?"  (troque o seller_id e a data)
SELECT seller_id, seller_city, seller_state, seller_zip_code_prefix, __START_AT, __END_AT
FROM olist.silver.sellers_historico
WHERE seller_id = '<seller_id>'
  AND TIMESTAMP '2026-10-01 00:00:00' >= __START_AT
  AND (TIMESTAMP '2026-10-01 00:00:00' < __END_AT OR __END_AT IS NULL);

-- um vendedor que se mudou, com as duas versões
SELECT seller_id, seller_city, seller_zip_code_prefix, __START_AT, __END_AT
FROM olist.silver.sellers_historico
WHERE seller_id IN (SELECT seller_id FROM olist.silver.sellers_historico GROUP BY 1 HAVING count(*) > 1 LIMIT 1)
ORDER BY __START_AT;
```

A regra do intervalo é **fechado no início e aberto no fim** (`>= __START_AT` e `< __END_AT`): o instante exato da mudança já pertence à versão nova, e nunca a duas versões ao mesmo tempo.

**Antes da carga inicial não há cadastro:** todo INSERT de vendedor tem `change_ts = 2026-09-06 00:00:00`; uma data anterior não retorna linha (e isso é correto).

Se `__START_AT`/`__END_AT` aparecerem com tipo diferente de `TIMESTAMP` ou o `DELETE` criar linha, avise.

**Etapa 3.2 (`customers`):** a tabela é nova; basta *Dry run* e *Run pipeline* **normal** (as demais tabelas não mudam). Não precisa mexer na lista de fontes.

```sql
SELECT count(*) AS versoes, count(DISTINCT customer_id) AS chaves,
       count_if(__END_AT IS NULL) AS abertas
FROM olist.silver.customers_historico;                     -- 101.008 · 98.476 · 98.188

SELECT count(*) FROM olist.silver.customers c
JOIN olist.silver.customers_historico h USING (customer_id)
WHERE c.operation = 'DELETE' AND h.__END_AT IS NULL;       -- 0

-- o cliente na data do pedido: cada pedido deve achar exatamente UMA versão
SELECT count(*) AS linhas,
       count(h.customer_id) AS com_versao,
       count(DISTINCT o.order_id) AS pedidos_distintos
FROM olist.silver.orders o
LEFT JOIN olist.silver.customers_historico h
  ON o.customer_id = h.customer_id
 AND o.order_purchase_timestamp >= h.__START_AT
 AND (o.order_purchase_timestamp < h.__END_AT OR h.__END_AT IS NULL);   -- 98.476 · 98.476 · 98.476

-- pedidos cujo CEP na data da compra difere do CEP atual do cliente
SELECT count(*) AS pedidos_com_cep_diferente_do_atual
FROM olist.silver.orders o
JOIN olist.silver.customers_historico h
  ON o.customer_id = h.customer_id
 AND o.order_purchase_timestamp >= h.__START_AT
 AND (o.order_purchase_timestamp < h.__END_AT OR h.__END_AT IS NULL)
JOIN olist.silver.customers c ON c.customer_id = o.customer_id
WHERE c.operation <> 'DELETE' AND h.customer_zip_code_prefix <> c.customer_zip_code_prefix;   -- até 2.526
```

O `INSERT` de cada cliente tem `change_ts` igual ao instante do pedido, por isso o `>=` no início do intervalo encontra versão para todos os pedidos, inclusive os 288 de clientes excluídos (a exclusão acontece depois da compra).
